# Lab Assignment 12
## Mixture of Experts inside a Transformer: routing, load balancing and capacity

---

### Objective

The tutorial built a mixture of experts on a toy: four clusters, four linear experts, and a router that
learned to send each cluster to its own expert. It ended on a failure — with an unlucky start, the
router overloaded one expert and the accuracy dropped.

This assignment moves MoE to where it is used in real language models: the **feed-forward layer of a
Transformer block**, with **every token routed separately**, exactly like the lecture's
*"Sparsely activated MoE Transformer block"* slide. The task is classifying news articles into four
topics.

You will build the layer yourself and then test four claims from the lecture:

1. Noisy top-k gating trains the router — **for every k?** (Task 2)
2. The **load-balancing loss** from the Switch Transformer spreads tokens evenly over the experts. (Task 3)
3. Uneven routing makes tokens **overflow an expert's capacity** and get dropped. (Task 4)
4. Experts **do not specialise by topic** (the Mixtral slides). (Task 5)

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Data | 2-D toy points | **AG News** articles: World, Sports, Business, Sci/Tech |
| Where the MoE sits | it is the whole model | the **feed-forward layer of a Transformer block** |
| What is routed | one point | **every token**, separately |
| Experts | 4 straight lines | **8** feed-forward networks |
| Router | softmax, top-1 | **noisy top-k** (Shazeer et al.), then the **Switch** router |
| Computation | all experts run, then 3 zeroed | only the chosen experts run (**sparse dispatch**) |
| Measured | accuracy, cluster table | router learning, **load balance**, **dropped tokens**, specialisation |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Build the MoE layer (noisy top-k gating, sparse dispatch) | **4** |
| **Task 2** | Train it — and find the top-1 trap | **4** |
| **Task 3** | The load-balancing loss | **4** |
| **Task 4** | Expert capacity and dropped tokens | **4** |
| **Task 5** | What did the experts learn? | **4** |
| | **Total** | **20** |

---

### Before you start

- **You need the file `mini_agnews.tsv`** in the same folder as this notebook: 16,000 training and
  2,000 test articles, 4,000 and 500 per topic. **In Colab:** click the folder icon in the left sidebar
  and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Use this exact file. Do not download AG News yourself — the numbers you compare depend on this
  sample, and it has been cleaned so that no test article also appears in the training set.
- A GPU is **not** required. Each model trains in roughly 10–40 seconds on a laptop or Colab CPU, and
  you train six of them.
- The **Setup** cell gives you the data pipeline, the Transformer block and the training loop, so
  your time goes on the MoE parts. Read it once: your layer must plug into it.
- Everything is seeded, so re-running gives the same numbers. Your numbers may still differ slightly
  from a classmate's if you write the layer differently — the **patterns** are what is marked.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

Run this once. It is given to you — do not change it.

## Environment check (dependencies and data file)

The lab needs only `torch` and `numpy` (both come pre-installed on Google Colab; the next cell installs them if they are missing) and the file `mini_agnews.tsv` next to the notebook. If the file is not there and you are on Colab, the cell below opens the upload dialog.

In [1]:
# Install the dependencies (no-op if they are already installed), then make sure the data file is present.
!pip install -q torch numpy

import os
if not os.path.exists('mini_agnews.tsv'):
    try:
        from google.colab import files          # only exists on Google Colab
        print('Please upload mini_agnews.tsv ...')
        files.upload()
    except ImportError:
        raise FileNotFoundError('mini_agnews.tsv must be in the same folder as this notebook.')
print('mini_agnews.tsv found:', os.path.exists('mini_agnews.tsv'))

mini_agnews.tsv found: True


In [2]:
import re, time, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

TOPICS    = ['World', 'Sports', 'Business', 'Sci/Tech']
MAX_LEN   = 64      # tokens kept per article
D_MODEL   = 64      # size of every token vector
D_HIDDEN  = 128     # hidden size inside each expert
N_EXPERTS = 8
EPOCHS    = 3
BATCH     = 64      # articles per batch

# ---------------- data ----------------
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]", text.lower())

rows = [line.rstrip('\n').split('\t') for line in open('mini_agnews.tsv', encoding='utf-8')][1:]
train_rows = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'train']
test_rows  = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'test']

counts = collections.Counter(w for toks, _ in train_rows for w in toks)
vocab = ['<pad>', '<unk>'] + [w for w, n in counts.most_common() if n >= 2]
word_index = {w: i for i, w in enumerate(vocab)}

def encode(data):
    X = torch.zeros(len(data), MAX_LEN, dtype=torch.long)          # 0 = <pad>, at the END
    for i, (toks, _) in enumerate(data):
        ids = [word_index.get(w, 1) for w in toks[:MAX_LEN]]       # 1 = <unk>
        X[i, :len(ids)] = torch.tensor(ids)
    return X, torch.tensor([label for _, label in data])

X_train, y_train = encode(train_rows)
X_test,  y_test  = encode(test_rows)
print(f"vocabulary {len(vocab)}   X_train {tuple(X_train.shape)}   X_test {tuple(X_test.shape)}")
print(f"real tokens per article: mean {(X_train != 0).sum(1).float().mean():.1f}")

# ---------------- model ----------------
class Expert(nn.Module):
    # One expert = the ordinary Transformer feed-forward network: Linear -> ReLU -> Linear.
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_model))
    def forward(self, x):
        return self.net(x)

class TinyTransformerClassifier(nn.Module):
    # Embedding -> one Transformer block (self-attention, then `ffn`) -> mean over tokens -> 4 topics.
    # `ffn` is either ONE Expert (the dense model) or your MoELayer. It receives only the REAL
    # tokens of the batch, flattened row by row into shape (T, D_MODEL).
    def __init__(self, vocab_size, ffn):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, D_MODEL, padding_idx=0)
        self.pos  = nn.Embedding(MAX_LEN, D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, num_heads=4, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.ffn  = ffn
        self.out  = nn.Linear(D_MODEL, len(TOPICS))

    def forward(self, X):
        mask = X != 0
        x = self.emb(X) + self.pos(torch.arange(X.shape[1]))
        a = self.ln1(x)
        x = x + self.attn(a, a, a, key_padding_mask=~mask, need_weights=False)[0]
        tokens = self.ln2(x)[mask]                 # (T, D_MODEL): real tokens only, row by row
        y = torch.zeros_like(x)
        y[mask] = self.ffn(tokens)
        x = x + y                                  # residual connection around the ffn
        return self.out((x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True))

def make_model(make_ffn, seed=42):
    torch.manual_seed(seed)
    return TinyTransformerClassifier(len(vocab), make_ffn())

# ---------------- training and evaluation ----------------
def train_model(model, alpha=0.0, seed=42):
    # Adam + cross-entropy. If alpha > 0 it also adds alpha * load_balancing_loss(...),
    # a function YOU write in Task 3.
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(X_train))
        for b in range(0, len(X_train), BATCH):
            idx = order[b:b + BATCH]
            loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
            if alpha > 0:
                f = model.ffn
                loss = loss + alpha * load_balancing_loss(f.probs, f.top1, f.n_experts)
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"  trained in {time.time() - start:.0f} s")
    return model

@torch.no_grad()
def evaluate(model):
    # Returns (test accuracy, routing). `routing` is a list with one tensor per test batch of
    # BATCH articles: the chosen experts of every real token, shape (tokens, k), best first.
    # None for a dense model.
    model.eval()
    correct, routing = 0, []
    for b in range(0, len(X_test), BATCH):
        logits = model(X_test[b:b + BATCH])
        correct += (logits.argmax(1) == y_test[b:b + BATCH]).sum().item()
        if hasattr(model.ffn, 'top_idx'):
            routing.append(model.ffn.top_idx.clone())
    return correct / len(X_test), (routing or None)

# ---------------- the dense baseline ----------------
print("\nDense baseline (one feed-forward network, no experts):")
dense = train_model(make_model(lambda: Expert(D_MODEL, D_HIDDEN)))
dense_acc, _ = evaluate(dense)
print(f"  test accuracy {dense_acc:.4f}")

vocabulary 19359   X_train (16000, 64)   X_test (2000, 64)
real tokens per article: mean 43.8

Dense baseline (one feed-forward network, no experts):


  trained in 22 s


  test accuracy 0.8505


---
# Task 1: Build the MoE layer &nbsp;&nbsp;*(4 marks)*

Two classes. The **router** decides, for each token, which experts to use and how much to trust each
one. The **MoE layer** sends each token to its chosen experts and adds up the weighted answers.

**The router — noisy top-k gating** (Shazeer et al. 2017, the *Noisy Top-K Gating* slides in part 1):

$$H(x)_i = (x\,W_g)_i + \text{StandardNormal}()\cdot\text{Softplus}\big((x\,W_{noise})_i\big)$$

$$\text{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the top } k \\ -\infty & \text{otherwise}\end{cases}
\qquad\qquad G(x) = \text{Softmax}\big(\text{KeepTopK}(H(x), k)\big)$$

The noise is added **only while training**. The $-\infty$ makes the softmax give exactly 0 to every
expert that was not chosen, so each row of $G(x)$ has exactly $k$ non-zero entries that add up to 1.

**The MoE layer — sparse dispatch.** For each token $t$:

$$y_t = \sum_{i\ \in\ \text{chosen}(t)} G(x_t)_i \; E_i(x_t)$$

Unlike the demo, an expert must run **only on the tokens routed to it**. That is the whole point of
a sparse layer: with $k=2$ of 8 experts, each token costs 2 experts' worth of compute, not 8.

### What your code must create

| Name | Is |
|---|---|
| `NoisyTopKRouter(d_model, n_experts, k)` | has `w_gate` and `w_noise` (bias-free `nn.Linear(d_model, n_experts)`); `forward(x)` returns `gates, top_idx, probs` |
| `gates` | shape `(T, n_experts)`: exactly `k` non-zeros per row, each row sums to 1 |
| `top_idx` | shape `(T, k)`: the chosen experts, best first |
| `probs` | shape `(T, n_experts)`: softmax over **all** experts of the logits $H(x)$ — Task 3 needs it |
| `MoELayer(d_model, d_hidden, n_experts, router)` | has `.router`, `.experts` (an `nn.ModuleList` of `n_experts` `Expert`s) and `.n_experts` |
| `MoELayer.forward(x)` | returns `y` with the same shape as `x`, and stores `self.probs`, `self.top_idx` and `self.top1` (= `top_idx[:, 0]`, each token's first choice) |

### Output your code must print

Test the layer on `x = torch.randn(10, D_MODEL)` with `k = 2`, in training mode:

```
gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576
```

### Hints

- `logits.topk(k, dim=-1)` gives both the top values and their indices.
- `torch.full_like(logits, float('-inf'))` then `.scatter(-1, top_idx, top_values)` builds the
  KeepTopK vector in one line.
- `self.training` is `True` after `model.train()` and `False` after `model.eval()`.
- `(gates[:, i] > 0).nonzero().squeeze(1)` gives the row indices of the tokens routed to expert `i`.
- Count parameters with `sum(p.numel() for p in module.parameters())`.

In [3]:
# ---------------- TASK 1 ----------------

class NoisyTopKRouter(nn.Module):
    def __init__(self, d_model, n_experts, k):
        super().__init__()
        # TODO 1: w_gate and w_noise -- bias-free Linear layers, d_model -> n_experts
        self.w_gate  = nn.Linear(d_model, n_experts, bias=False)
        self.w_noise = nn.Linear(d_model, n_experts, bias=False)
        # TODO 2: store k (and n_experts)
        self.k = k
        self.n_experts = n_experts

    def forward(self, x):
        # TODO 3: clean logits = x W_g
        logits = self.w_gate(x)                                           # (T, n_experts)
        # TODO 4: training mode only: add StandardNormal * Softplus(x W_noise)
        if self.training:
            logits = logits + torch.randn_like(logits) * F.softplus(self.w_noise(x))
        # TODO 5: KeepTopK -- keep the k largest logits per row, everything else = -inf
        top_vals, top_idx = logits.topk(self.k, dim=-1)                   # both (T, k), best first
        kept = torch.full_like(logits, float('-inf')).scatter(-1, top_idx, top_vals)
        # TODO 6: gates = softmax over the kept logits (exactly 0 for the others);
        #         probs = softmax over ALL logits (needed by the load-balancing loss in Task 3)
        gates = F.softmax(kept, dim=-1)
        probs = F.softmax(logits, dim=-1)
        # TODO 7
        return gates, top_idx, probs


class MoELayer(nn.Module):
    def __init__(self, d_model, d_hidden, n_experts, router):
        super().__init__()
        # TODO 8: router, n_experts and n_experts independent feed-forward experts
        self.router = router
        self.n_experts = n_experts
        self.experts = nn.ModuleList([Expert(d_model, d_hidden) for _ in range(n_experts)])

    def forward(self, x):
        # TODO 9: route every token and remember what the later tasks need
        gates, top_idx, probs = self.router(x)
        self.probs, self.top_idx, self.top1 = probs, top_idx, top_idx[:, 0]
        # TODO 10
        y = torch.zeros_like(x)
        # TODO 11: SPARSE dispatch -- expert i only runs on the tokens that chose it
        for i, expert in enumerate(self.experts):
            rows = (top_idx == i).any(dim=1).nonzero().squeeze(1)          # tokens routed to expert i (gate > 0)
            if rows.numel() == 0:
                continue
            y = y.index_add(0, rows, gates[rows, i].unsqueeze(1) * expert(x[rows]))
        # TODO 12
        return y


# TODO 13: build the layer (k = 2), run it in training mode on 10 random tokens and print the checks
moe = MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2))
moe.train()
x = torch.randn(10, D_MODEL)
gates, top_idx, probs = moe.router(x)
y = moe(x)
print('gates shape:', tuple(gates.shape))
print('non-zeros per row:', (gates > 0).sum(1).tolist())
print('row sums: all 1.0 ->', torch.allclose(gates.sum(1), torch.ones(10)))
print('output shape:', tuple(y.shape))

# TODO 14: parameter counts
n_params = lambda module: sum(p.numel() for p in module.parameters())
print()
print(f'parameters in the MoE layer: {n_params(moe):,}')
print(f'active per token (router + 2 experts): {n_params(moe.router) + 2 * n_params(moe.experts[0]):,}')
print(f'dense feed-forward layer: {n_params(Expert(D_MODEL, D_HIDDEN)):,}')

gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576


---
# Task 2: Train it — and find the top-1 trap &nbsp;&nbsp;*(4 marks)*

Now put the layer inside the Transformer and train it. The Switch Transformer slides make a big claim
for **top-1** routing: one expert per token is enough, and it is cheaper than two. So train the noisy
top-k router twice — once with `k = 2`, once with `k = 1` — and watch **whether the router learns
anything**.

To see that, copy the router's weights **before** training and measure how far they moved:

$$\text{router moved} = \lVert W_g^{\text{after}} - W_g^{\text{before}} \rVert$$

Then build a third router, the **Switch router** (Fedus et al. 2021), which routes to one expert in a
different way:

$$p(x) = \text{Softmax}(x\,W_g) \qquad i^* = \arg\max_i\, p(x)_i \qquad y = p(x)_{i^*}\,E_{i^*}(x)$$

No noise, and the gate value is the **softmax probability over all experts** — not a softmax over just
the one kept logit. This is the same top-1 rule as in the tutorial's section 3.

### What your code must create

| Name | Is |
|---|---|
| `SwitchRouter(d_model, n_experts)` | has `w_gate`; same `forward` interface as `NoisyTopKRouter` (with `k = 1`) |
| `noisy_k2`, `noisy_k1`, `switch_a0` | three trained `TinyTransformerClassifier`s with `MoELayer` ffns, all built with `make_model(..., seed=42)` and trained with `train_model(model)` |
| `usage(routing)` | a function: share of test tokens whose **first choice** is each expert, as a length-8 array |

### Output your code must print

```
MODEL           test acc   router moved   expert usage (share of test tokens)
dense FFN        0.____         -          -
noisy top-2      0.____       _.___        [0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__]
noisy top-1      0.____       _.___        [ ... ]
switch top-1     0.____       _.___        [ ... ]
```

### Then answer

**Q1.** One of the three routers did not move **at all**. Which one, and why? Use the gradient from the
lecture's backpropagation slide, $\nabla L / \nabla G(x)_i = E_i(x)^T\, \nabla L / \nabla y$, and work out
what $G(x)$ is for the chosen expert when $k = 1$. Why does the Switch router not have this problem?
Finally: that model's accuracy is not much worse than the others — what does that tell you about
this task?

### Hints

- `W0 = model.ffn.router.w_gate.weight.detach().clone()` **before** `train_model(...)`.
- `torch.cat(routing)[:, 0]` gives every test token's first-choice expert;
  `np.bincount(..., minlength=8)` counts each expert.
- In `SwitchRouter`, `top_idx` must still have shape `(T, 1)` so that `MoELayer` works unchanged.

In [4]:
# ---------------- TASK 2 ----------------

class SwitchRouter(nn.Module):
    def __init__(self, d_model, n_experts):
        super().__init__()
        # TODO 15: w_gate (bias-free Linear); k is always 1
        self.w_gate = nn.Linear(d_model, n_experts, bias=False)
        self.k = 1
        self.n_experts = n_experts

    def forward(self, x):
        # TODO 16: probs = softmax over ALL experts (no noise)
        probs = F.softmax(self.w_gate(x), dim=-1)
        # TODO 17: top_idx = argmax, kept as shape (T, 1) so that MoELayer works unchanged
        top_idx = probs.argmax(dim=-1, keepdim=True)
        # TODO 18: gate = the softmax probability of the chosen expert (NOT a softmax over one value)
        gates = torch.zeros_like(probs).scatter(-1, top_idx, probs.gather(-1, top_idx))
        # TODO 19
        return gates, top_idx, probs


def usage(routing):
    # TODO 20: share of test tokens whose FIRST choice is each expert
    first = torch.cat(routing)[:, 0]
    return np.bincount(first.numpy(), minlength=N_EXPERTS) / len(first)


# TODO 21: build, copy the router weights BEFORE training, train, measure how far w_gate moved
results = {}
def train_and_measure(name, make_ffn, alpha=0.0):
    print(f'training {name} (alpha={alpha}) ...')
    model = make_model(make_ffn, seed=42)
    W0 = model.ffn.router.w_gate.weight.detach().clone()               # copy BEFORE training
    train_model(model, alpha=alpha)
    moved = (model.ffn.router.w_gate.weight.detach() - W0).norm().item()
    acc, routing = evaluate(model)
    results[name] = dict(model=model, acc=acc, moved=moved, usage=usage(routing), routing=routing)
    return model

noisy_k2  = train_and_measure('noisy top-2',  lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2)))
noisy_k1  = train_and_measure('noisy top-1',  lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=1)))
switch_a0 = train_and_measure('switch top-1', lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, SwitchRouter(D_MODEL, N_EXPERTS)))

# TODO 22: comparison table (dense baseline first)
np.set_printoptions(precision=2, suppress=True)
print()
print(f"{'MODEL':<14}{'test acc':>9}{'router moved':>14}   expert usage (share of test tokens)")
print(f"{'dense FFN':<14}{dense_acc:>9.4f}{'-':>14}   -")
for label, key in [('noisy top-2', 'noisy top-2'), ('noisy top-1', 'noisy top-1'), ('switch top-1', 'switch top-1')]:
    r = results[key]
    print(f"{label:<14}{r['acc']:>9.4f}{r['moved']:>14.3f}   {r['usage']}")

training noisy top-2 (alpha=0.0) ...


  trained in 34 s


training noisy top-1 (alpha=0.0) ...


  trained in 30 s


training switch top-1 (alpha=0.0) ...


  trained in 33 s



MODEL          test acc  router moved   expert usage (share of test tokens)
dense FFN        0.8505             -   -
noisy top-2      0.8415         1.431   [0.07 0.1  0.14 0.11 0.27 0.15 0.1  0.05]
noisy top-1      0.8475         0.000   [0.1  0.16 0.09 0.08 0.14 0.17 0.1  0.16]
switch top-1     0.8385         1.513   [0.15 0.11 0.12 0.16 0.13 0.11 0.13 0.09]


---
# Task 3: The load-balancing loss &nbsp;&nbsp;*(4 marks)*

The tutorial's section 5 ended with a router that overloaded one expert. Look at your Task 2 table:
the **noisy top-2** router is the uneven one — compare its busiest and quietest experts with an even
share of $1/8 = 0.125$. The Switch Transformer's fix is an extra loss term (the *Load Balancing Loss*
slide in part 2). For a batch with $T$ tokens and $N$ experts:

$$f_i = \frac{1}{T}\sum_{x \in \text{batch}} \mathbb{1}\{\arg\max p(x) = i\}
\qquad\quad P_i = \frac{1}{T}\sum_{x \in \text{batch}} p_i(x)
\qquad\quad \text{loss} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i\,P_i$$

$f_i$ is the **fraction of tokens dispatched** to expert $i$ (with top-2 routing, count each token's
**first** choice — that is `self.top1`); $P_i$ is the **average router probability** for expert $i$.
When every expert gets exactly $1/N$ of the tokens, the sum is exactly 1. `train_model` already adds
`alpha * load_balancing_loss(...)` when `alpha > 0` — you write the function.

### What your code must create

| Name | Is |
|---|---|
| `load_balancing_loss(probs, top1, n_experts)` | returns the scalar $N \sum_i f_i P_i$ (without $\alpha$) |
| `noisy_k2_a001`, `noisy_k2_a01` | **noisy top-2** models trained with `alpha=0.01` and `alpha=0.1` (`noisy_k2` from Task 2 is the `alpha=0` one) |

### Output your code must print

First two sanity checks on made-up routings with 8 experts:

```
perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000
```

Then the comparison, where **usage CV** is the coefficient of variation of the usage array
(standard deviation ÷ mean: 0 means perfectly even):

```
alpha    test acc   usage CV   busiest expert   quietest expert
0.00      0.____     0.___        0.___           0.___
0.01      0.____     0.___        0.___           0.___
0.10      0.____     0.___        0.___           0.___
```

### Then answer

**Q2.** Did the loss even out the routing, and what did it cost in accuracy? $f_i$ is computed with an
argmax, so no gradient flows through it — so which part of the loss actually trains the router, and
why does multiplying by $f_i$ still push tokens **away** from busy experts?

### Hints

- `F.one_hot(top1, n_experts).float().mean(0)` gives $f$; `probs.mean(0)` gives $P$.
- For the balanced check: 8 tokens, token $t$ sent to expert $t$, and `probs` all equal to $1/8$.
- Train with `train_model(model, alpha=...)`, building each model with `make_model(..., seed=42)`.

In [5]:
# ---------------- TASK 3 ----------------

def load_balancing_loss(probs, top1, n_experts):
    # TODO 23: f_i = fraction of tokens whose first choice is expert i  (no gradient: argmax)
    f = F.one_hot(top1, n_experts).float().mean(0)
    # TODO 24: P_i = mean router probability of expert i  (this is what carries the gradient)
    P = probs.mean(0)
    # TODO 25
    return n_experts * (f * P).sum()


# TODO 26: sanity check 1 -- token t goes to expert t, probabilities all 1/8   -> 1.000
probs_bal = torch.full((8, 8), 1 / 8)
print(f'perfectly balanced: {load_balancing_loss(probs_bal, torch.arange(8), 8).item():.3f}')
# TODO 27: sanity check 2 -- everything to expert 0 with probability 1         -> 8.000
probs_one = torch.zeros(8, 8); probs_one[:, 0] = 1.0
print(f'everything to expert 0 (probability 1): {load_balancing_loss(probs_one, torch.zeros(8, dtype=torch.long), 8).item():.3f}')

# TODO 28: noisy top-2 models with the balancing loss
print()
noisy_k2_a001 = train_and_measure('noisy top-2 a=0.01', lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2)), alpha=0.01)
noisy_k2_a01  = train_and_measure('noisy top-2 a=0.1',  lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2)), alpha=0.1)

# TODO 29: table  (usage CV = standard deviation / mean of the usage array; 0 = perfectly even)
cv = lambda u: u.std() / u.mean()
print()
print(f"{'alpha':<8}{'test acc':>9}{'usage CV':>10}{'busiest expert':>16}{'quietest expert':>17}")
for alpha, key in [(0.00, 'noisy top-2'), (0.01, 'noisy top-2 a=0.01'), (0.10, 'noisy top-2 a=0.1')]:
    r = results[key]; u = r['usage']
    print(f"{alpha:<8.2f}{r['acc']:>9.4f}{cv(u):>10.3f}{u.max():>16.3f}{u.min():>17.3f}")

perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000

training noisy top-2 a=0.01 (alpha=0.01) ...


  trained in 35 s


training noisy top-2 a=0.1 (alpha=0.1) ...


  trained in 37 s



alpha    test acc  usage CV  busiest expert  quietest expert
0.00       0.8415     0.492           0.267            0.055
0.01       0.8405     0.268           0.199            0.081
0.10       0.8455     0.142           0.164            0.103


---
# Task 4: Expert capacity and dropped tokens &nbsp;&nbsp;*(4 marks)*

Why does balance matter if accuracy barely moves? Because real MoE models put each expert on its own
device, and every device needs a **fixed-size** buffer (the *How to set Expert Capacity?* slides in
part 2). For a batch of $T$ tokens:

$$\text{capacity} = \left\lfloor \text{capacity factor} \times \frac{T \cdot k}{N} \right\rfloor$$

($T \cdot k$ because with top-$k$ routing every token takes up $k$ expert slots.) If more tokens than
that are routed to an expert, the extra ones **overflow**: they are **dropped** from that expert, skip
it entirely, and continue only through the residual connection. A capacity factor of 1.0 leaves no
slack at all; Switch Transformer used 1.25.

You do not need to change the model. `evaluate` already returns the routing of each test batch; count
how many tokens would overflow.

### What your code must create

| Name | Is |
|---|---|
| `dropped_fraction(routing, n_experts, capacity_factor)` | fraction of all token-to-expert assignments that overflow, computed **batch by batch** |

### Output your code must print

A check you can do by hand (the slides' example: 6 tokens, 3 experts, capacity factor 1.0):

```
toy check: 0.167
```

Then, for the noisy top-2 models trained with `alpha = 0` and `alpha = 0.1`:

```
capacity factor   dropped (alpha=0.00)   dropped (alpha=0.10)
1.00                    _._%                   _._%
1.25                    _._%                   _._%
1.50                    _._%                   _._%
2.00                    _._%                   _._%
```

### Then answer

**Q3.** The balanced model still drops tokens at capacity factor 1.0 — why, if its overall usage is
nearly even? What happens to a dropped token, and why is that not a disaster for accuracy here? Why not
just set the capacity factor to 2.0?

### Hints

- For the toy check use `routing = [torch.tensor([[0], [0], [0], [1], [2], [2]])]` — 6 tokens, $k = 1$,
  3 experts: capacity is $\lfloor 1.0 \times 6 \cdot 1/3 \rfloor = 2$, expert 0 receives 3, so one
  assignment in six overflows.
- Each routing tensor has shape `(T, k)`. `torch.bincount(r.flatten(), minlength=n_experts)` counts
  every assignment per expert; `(counts - capacity).clamp(min=0).sum()` counts the overflow.

In [6]:
# ---------------- TASK 4 ----------------

def dropped_fraction(routing, n_experts, capacity_factor):
    dropped, total = 0, 0
    for r in routing:                                                       # capacity is a PER-BATCH limit
        T, k = r.shape
        # TODO 30: capacity = floor(capacity_factor * T * k / n_experts)
        capacity = int(capacity_factor * T * k / n_experts)
        # TODO 31: assignments beyond capacity, per expert, summed over experts and batches
        counts = torch.bincount(r.flatten(), minlength=n_experts)
        dropped += (counts - capacity).clamp(min=0).sum().item()
        total += T * k
    # TODO 32
    return dropped / total


# TODO 33: toy check from the slides (6 tokens, 3 experts, capacity factor 1.0)
toy = [torch.tensor([[0], [0], [0], [1], [2], [2]])]
print(f'toy check: {dropped_fraction(toy, 3, 1.0):.3f}')

# TODO 34: dropped fraction vs capacity factor, alpha = 0 and alpha = 0.1  (noisy top-2 models)
routing_a0  = results['noisy top-2']['routing']
routing_a01 = results['noisy top-2 a=0.1']['routing']
print()
print(f"{'capacity factor':<17}{'dropped (alpha=0.00)':>22}{'dropped (alpha=0.10)':>23}")
for cf in [1.0, 1.25, 1.5, 2.0]:
    print(f"{cf:<17.2f}{100 * dropped_fraction(routing_a0, N_EXPERTS, cf):>21.1f}%{100 * dropped_fraction(routing_a01, N_EXPERTS, cf):>22.1f}%")

toy check: 0.167

capacity factor    dropped (alpha=0.00)   dropped (alpha=0.10)
1.00                               9.9%                   5.8%
1.25                               3.7%                   0.2%
1.50                               0.8%                   0.0%
2.00                               0.0%                   0.0%


In [7]:
# Extra diagnostic (not required by the task): WHY does a nearly balanced model still overflow at capacity factor 1.0?
# Compare the real dropped share with a simulation in which every assignment is routed independently at random,
# (i) with the model's own average usage and (ii) with a perfectly even usage of 1/8 per expert.
def simulated_drop(routing, cf, marginal, trials=10):
    out = []
    for s in range(trials):
        g = torch.Generator().manual_seed(s)
        fake = [torch.multinomial(marginal, r.numel(), replacement=True, generator=g).view(-1, 1) for r in routing]
        out.append(dropped_fraction(fake, N_EXPERTS, cf))
    return float(np.mean(out))

print(f"{'model':<12}{'factor':>7}{'real routing':>14}{'independent, own usage':>25}{'independent, even usage':>25}")
for name, rt in [('alpha=0.00', routing_a0), ('alpha=0.10', routing_a01)]:
    own = torch.bincount(torch.cat(rt).flatten(), minlength=N_EXPERTS).float(); own = own / own.sum()
    even = torch.full((N_EXPERTS,), 1 / N_EXPERTS)
    for cf in [1.0, 1.25]:
        print(f"{name:<12}{cf:>7.2f}{100 * dropped_fraction(rt, N_EXPERTS, cf):>13.1f}%"
              f"{100 * simulated_drop(rt, cf, own):>24.1f}%{100 * simulated_drop(rt, cf, even):>24.1f}%")
print(f"\ntokens per test batch: about {int(np.mean([len(r) for r in routing_a0]))}  (each takes 2 expert slots with top-2 routing)")

model        factor  real routing   independent, own usage  independent, even usage
alpha=0.00     1.00          9.9%                     9.3%                     1.5%


alpha=0.00     1.25          3.7%                     3.6%                     0.0%
alpha=0.10     1.00          5.8%                     5.3%                     1.5%


alpha=0.10     1.25          0.2%                     0.1%                     0.0%

tokens per test batch: about 2736  (each takes 2 expert slots with top-2 routing)


---
# Task 5: What did the experts learn? &nbsp;&nbsp;*(4 marks)*

The Mixtral slides in part 2 ask whether experts specialise — does one expert handle the maths, another
the code? Their answer for Mixtral was **no**: experts did not specialise by domain, but consecutive
tokens were often sent to the same expert. Test both on your own model.

Use the **`alpha = 0.1` noisy top-2 model** (`noisy_k2_a01`) and each token's **first-choice**
expert.

### What your code must create and print

**(a) Topic × expert table.** For each topic, the share of **that topic's tokens** sent to each expert
(each row adds up to 1). Then each topic's favourite expert and its share — compare with an even
spread, $1/8 = 0.125$.

```
            E0    E1    E2    E3    E4    E5    E6    E7
World      0.__  0.__  ...
Sports     ...
Business   ...
Sci/Tech   ...

favourite expert:  World E_ (0.__)   Sports E_ (0.__)   Business E_ (0.__)   Sci/Tech E_ (0.__)
```

**(b) Top tokens per expert.** The 10 most frequent tokens routed to each expert.

```
E0 (0.__ of tokens): ['.', 'the', ...]
...
```

**(c) Consecutive tokens.** Of all pairs of neighbouring tokens **within the same article**, the
fraction routed to the same expert — and what chance would give, $\sum_i u_i^2$ where $u$ is the usage
array.

```
consecutive tokens on the same expert: 0.___   (chance: 0.___)
```

### Then answer

**Q4.** Do your experts specialise by topic? Use the table: is any expert a "Sports expert"? What do the
top-token lists suggest the router actually groups by? Compare with the Mixtral slide.

**Q5.** Consecutive tokens share an expert more often than chance. Why might that be, given what the
self-attention layer **before** the MoE layer does to each token's vector?

### Hints

- Take each routing tensor's first column, `r[:, 0]`. The rows follow `X_batch[X_batch != 0]` exactly — the same row-by-row order the model
  uses — so that expression gives the token ids that line up with each routing tensor.
- `torch.repeat_interleave(y_batch, (X_batch != 0).sum(1))` gives each token its article's topic.
- Padding is at the **end** of every article, so each article's tokens are one consecutive run in the
  routing tensor; `(X_batch != 0).sum(1)` gives the run lengths.

In [8]:
# ---------------- TASK 5 ----------------

# TODO 35: token ids and topics, lined up with the first column of the routing tensors of noisy_k2_a01
acc_a01, routing = evaluate(noisy_k2_a01)
tok_ids, tok_topic, art_lens = [], [], []
for b in range(0, len(X_test), BATCH):
    Xb, yb = X_test[b:b + BATCH], y_test[b:b + BATCH]
    real = Xb != 0
    tok_ids.append(Xb[real])                                               # same row-by-row order as the model
    tok_topic.append(torch.repeat_interleave(yb, real.sum(1)))             # each token gets its article's topic
    art_lens.append(real.sum(1))                                           # run length of every article
first = [r[:, 0] for r in routing]                                         # first-choice expert of every token
assert all(len(f) == len(t) for f, t in zip(first, tok_ids))               # alignment check
first_all, ids_all, topic_all = torch.cat(first), torch.cat(tok_ids), torch.cat(tok_topic)

# TODO 36: (a) topic x expert table (each row sums to 1) and the favourite expert of each topic
table = np.zeros((len(TOPICS), N_EXPERTS))
for t in range(len(TOPICS)):
    table[t] = np.bincount(first_all[topic_all == t].numpy(), minlength=N_EXPERTS) / (topic_all == t).sum().item()
print('(a) share of each topic\'s tokens sent to each expert (first choice)\n')
print(' ' * 10 + ''.join(f'{"E" + str(e):>6}' for e in range(N_EXPERTS)))
for t, name in enumerate(TOPICS):
    print(f'{name:<10}' + ''.join(f'{v:>6.2f}' for v in table[t]))
print('\nfavourite expert:  ' + '   '.join(f'{name} E{table[t].argmax()} ({table[t].max():.2f})' for t, name in enumerate(TOPICS)))
print(f'(an even spread would be {1 / N_EXPERTS:.3f} per expert)')

# TODO 37: (b) the 10 most frequent tokens of every expert
print('\n(b) most frequent tokens per expert\n')
u = usage(routing)
for e in range(N_EXPERTS):
    c = collections.Counter(ids_all[first_all == e].tolist())
    print(f'E{e} ({u[e]:.2f} of tokens): {[vocab[i] for i, _ in c.most_common(10)]}')

# TODO 38: (c) neighbouring tokens inside the same article routed to the same expert
same, pairs, offset_unused = 0, 0, 0
for f, lens in zip(first, art_lens):
    article = torch.repeat_interleave(torch.arange(len(lens)), lens)       # which article every token belongs to
    valid = article[1:] == article[:-1]                                    # only pairs INSIDE one article
    same += ((f[1:] == f[:-1]) & valid).sum().item()
    pairs += valid.sum().item()
chance = (u ** 2).sum()
print(f'\n(c) consecutive tokens on the same expert: {same / pairs:.3f}   (chance: {chance:.3f})')

(a) share of each topic's tokens sent to each expert (first choice)

              E0    E1    E2    E3    E4    E5    E6    E7
World       0.11  0.08  0.09  0.25  0.13  0.06  0.08  0.20
Sports      0.22  0.11  0.03  0.06  0.31  0.05  0.04  0.18
Business    0.08  0.17  0.24  0.09  0.08  0.17  0.12  0.06
Sci/Tech    0.14  0.13  0.05  0.04  0.13  0.24  0.23  0.04

favourite expert:  World E3 (0.25)   Sports E4 (0.31)   Business E2 (0.24)   Sci/Tech E5 (0.24)
(an even spread would be 0.125 per expert)

(b) most frequent tokens per expert

E0 (0.14 of tokens): ['the', 'to', ',', '.', 'a', 'for', '(', 'his', 'of', 'game']
E1 (0.12 of tokens): ['.', 'and', '-', 'ap', 'by', 'in', '(', 'it', 'at', 's']
E2 (0.10 of tokens): ['a', 'for', 'the', ',', '/', 'to', '-', 'its', 'us', 'oil']
E3 (0.11 of tokens): [',', '<unk>', 'of', 's', 'and', 'said', 'on', 'afp', 'iraq', 'over']
E4 (0.16 of tokens): ['<unk>', '-', 'to', ',', "'", 'and', '.', 'a', 'of', 'has']
E5 (0.13 of tokens): ['.', ',', 'in', 'as

In [9]:
# ---------------- Pattern check (re-run safety net) ----------------
# The five answers below describe PATTERNS. This cell checks that each pattern is present in the numbers above.
r = results
checks = {
    'Q1  noisy top-1 router did not move at all':            r['noisy top-1']['moved'] == 0.0,
    'Q1  noisy top-2 and Switch routers did move':           r['noisy top-2']['moved'] > 0 and r['switch top-1']['moved'] > 0,
    'Q2  balancing loss lowers the usage CV (0 -> 0.1)':     cv(r['noisy top-2 a=0.1']['usage']) < cv(r['noisy top-2']['usage']),
    'Q3  dropped share falls as capacity factor grows':      dropped_fraction(routing_a0, 8, 2.0) < dropped_fraction(routing_a0, 8, 1.0),
    'Q3  balanced model drops fewer tokens at factor 1.0':   dropped_fraction(routing_a01, 8, 1.0) < dropped_fraction(routing_a0, 8, 1.0),
    'Q5  neighbouring tokens share an expert > chance':      same / pairs > chance,
}
for name, ok in checks.items():
    print(('PASS  ' if ok else 'CHECK ') + name)

PASS  Q1  noisy top-1 router did not move at all
PASS  Q1  noisy top-2 and Switch routers did move
PASS  Q2  balancing loss lowers the usage CV (0 -> 0.1)
PASS  Q3  dropped share falls as capacity factor grows
PASS  Q3  balanced model drops fewer tokens at factor 1.0
PASS  Q5  neighbouring tokens share an expert > chance


---
## Your answers

*All numbers below are read from the outputs printed in the cells above (seed 42, 2,000 test articles). One accuracy point is 20 test articles, and the standard error of a test accuracy near 0.85 on 2,000 articles is about 0.8 points, so differences of one point or less are within noise.*

**Q1.** *(Task 2)*

**Which router did not move?** The **noisy top-1** router: its `router moved` value is exactly **0.000**, while the noisy top-2 router moved by 1.431 and the Switch router by 1.513.

**Why.** The router can only learn through the gate values G(x), because the gradient reaching it is ∇L/∇G(x)ᵢ = Eᵢ(x)ᵀ ∇L/∇y, which is then multiplied by ∂G/∂(logit). With k = 1, KeepTopK keeps a single logit and sets all the others to −∞. The softmax over one kept value is always exactly 1, whatever the value of that logit. So G(x) for the chosen expert is the constant 1 and all other gates are the constant 0. The derivative ∂G/∂logit is therefore 0, the gradient reaching `w_gate` is 0 (and `w_noise` gets none either, because the noise is added before the same softmax), and Adam receives nothing to update. The router stays at its random initialisation: it is a fixed random projection that never learns which expert is good for which token. With k = 2 the softmax is over two values, so the two gates trade off against each other (G₁ = 1 − G₂) and the gradient is non-zero.

**Why the Switch router does not have the problem.** It also routes to one expert, but its gate is the softmax probability over **all 8 experts**, p(x)ᵢ*, not a softmax over the single kept logit. That probability changes when the chosen logit changes relative to the other seven logits (∂pᵢ*/∂logitᵢ* = pᵢ*(1 − pᵢ*) and ∂pᵢ*/∂logitⱼ = −pᵢ* pⱼ), so the output y = p(x)ᵢ* Eᵢ*(x) depends on the router weights and the gradient flows. The Switch router moved by 1.513.

**What the accuracy tells us.** The router that never learned still gets 0.8475, compared with 0.8505 for the dense model, 0.8415 for noisy top-2 and 0.8385 for Switch. All four are within about 1.2 points of each other, which is within test noise. The task (assigning an article to one of four topics) is easy: a few topic words and a mean over about 44 tokens are enough, and the embeddings, the attention layer and the experts themselves still learn even when the routing is a fixed random split. So **test accuracy cannot reveal a broken router** on this task, and diagnostics such as "did the router weights move?" and "how are tokens spread over the experts?" are needed instead. It also shows that an MoE layer brings no accuracy advantage over the dense feed-forward layer here (the MoE layer has about 8 times the parameters of the dense feed-forward layer for the same accuracy).

**Q2.** *(Task 3)*

**Did the loss even out the routing?** Yes, and the stronger the weight the more even it became. The coefficient of variation of the expert usage fell from **0.492** (α = 0) to **0.268** (α = 0.01) and **0.142** (α = 0.1). The busiest expert's share fell from 0.267 to 0.199 to 0.164, and the quietest expert's share rose from 0.055 to 0.081 to 0.103 (an even share is 0.125).

**Cost in accuracy.** None that we can measure: 0.8415 (α = 0), 0.8405 (α = 0.01) and 0.8455 (α = 0.1). The differences are 0.1–0.4 points, well inside the ±0.8 point noise, so on this easy task the balancing is essentially free. (Its benefit shows up in Task 4, not in accuracy.)

**Which part of the loss trains the router.** Only **Pᵢ**, the mean router probability of expert i. It is the only differentiable quantity: fᵢ is computed with an argmax over the tokens, so it has zero gradient and acts as a constant per-expert weight. The gradient of the loss with respect to Pᵢ is proportional to fᵢ, so gradient descent lowers the probability of expert i by an amount proportional to how many tokens that expert currently receives. A busy expert (large fᵢ) is pushed down hard and a rarely used expert (small fᵢ) is barely pushed. Because the probabilities of each token sum to 1, probability mass moves from the overloaded experts to the others, the argmax for more and more tokens flips to the quieter experts, fᵢ changes, and the loss reaches its minimum value of 1 exactly when every expert gets 1/N of the tokens and has probability 1/N (the two sanity checks show 1.000 for perfect balance and 8.000 for total collapse onto one expert).

**Q3.** *(Task 4)*

**Why does the balanced model still drop tokens at capacity factor 1.0?** At factor 1.0 the capacity equals exactly the average load, ⌊T·k/N⌋ (a test batch has about 2,736 tokens, so with top-2 routing about 684 slots per expert), which leaves no slack: an expert that receives even slightly more than its fair share overflows. Two things cause this for the balanced model, and the extra diagnostic cell above separates them. **(1) The average usage is nearly even, but not perfectly:** with α = 0.1 the usage CV is still 0.142 and the busiest expert takes 0.164 instead of 0.125. This is the main cause: when every assignment is routed independently at random with the model's own average usage, 5.3 % overflow at factor 1.0, almost the same as the real 5.8 %. **(2) Chance alone:** even with a perfectly even usage of 1/8, the load of an expert in a single batch fluctuates around its mean, and capacity is a per-batch limit, which still gives 1.5 % overflow at factor 1.0 (and 0.0 % at 1.25). The clumping of neighbouring tokens on the same expert (Q5) adds only about half a point (5.8 % real vs. 5.3 % independent). The benefit of balancing is clear in the table: at factor 1.0 the overflow is 5.8 % against 9.9 % for the unbalanced model, at 1.25 it is 0.2 % against 3.7 %, and at 1.5 it is 0.0 % against 0.8 %.

**What happens to a dropped token, and why is it not a disaster?** A dropped token is not processed by that expert: its contribution from that expert is simply missing and the token continues only through the **residual connection**, i.e. it keeps the vector it had after self-attention (if both its choices are dropped, it skips the feed-forward layer completely). Here this barely matters, because the classifier averages over about 44 tokens per article, so a few per cent of tokens that miss one expert change the average very little, and the task is easy (accuracy is almost the same for every model).

**Why not just use a capacity factor of 2.0?** Because the capacity is a buffer that every expert must allocate in memory and compute whether it is filled or not. A factor of 2.0 doubles the buffer (and the all-to-all communication between devices) and wastes half of it on empty slots, which throws away the efficiency that makes sparse models attractive. It also hides the problem instead of fixing it: in the table the unbalanced model needs 2.0 to reach 0.0 % drops, while the balanced model is already at 0.2 % with the cheaper 1.25 (the value used by the Switch Transformer) and at 0.0 % with 1.5. A small capacity factor combined with a balancing loss is the better trade-off.

**Q4.** *(Task 5)*

**Do the experts specialise by topic? Only weakly.** In the topic × expert table every topic has a different favourite expert, and it receives about two times its even share: World → E3 (0.25), Sports → E4 (0.31), Business → E2 (0.24), Sci/Tech → E5 (0.24), against 0.125 for an even spread. But nothing is exclusive: even the strongest case (Sports → E4) sends only 31 % of the Sports tokens there, 69 % go to the other seven experts, and every expert receives tokens of every topic (a row's smallest entry is 0.03–0.06, its largest ones 0.24–0.31). So there is a statistical tilt, but there is no "Sports expert" in the sense of an expert that handles Sports and nothing else.

**What does the router group by?** The top-token lists show that the experts are mostly defined by **kinds of tokens**, not topics. Almost every list is dominated by function words and punctuation ('the', 'to', ',', '.', 'a', 'of', 'in', '-', '(' ...), different experts take different mixes of them (E0 'the', 'to', 'a', 'for'; E1 '.', 'and', '-'; E7 'in', "'", ')'), E4, E3, E5 and E6 take many rare words (`<unk>`), and some experts pick up news-agency markers (E1 'ap', E3 'afp', E6 'reuters'). A few topic-bearing words appear as well (E0 'game', E2 'oil', E3 'iraq'), and these explain the mild topic tilt in the table: when a topic's frequent content words happen to be routed to one expert, that expert becomes that topic's favourite. The balancing loss with α = 0.1 may also work against specialisation, because it pushes every expert towards receiving a similar number of tokens of the whole corpus.

**Comparison with Mixtral.** This agrees with the Mixtral slide: the Mixtral authors found no clear specialisation of experts by domain (the distribution of experts over, e.g., different subjects was similar), and the router instead followed **syntactic and positional** properties of the tokens, with consecutive tokens often going to the same expert. Our model shows the same two things (no clean topic experts, token-type-based groups, neighbouring tokens sharing experts: Q5). The topic tilt here is larger than in Mixtral, probably because our model is tiny, has only four very distinct topics, and a small vocabulary in which a few words (game, oil, iraq) are strong topic markers.

**Q5.** *(Task 5)*

**Observation.** 0.189 of neighbouring token pairs inside an article share the same first-choice expert, against 0.128 expected by chance (Σuᵢ² of the usage array), which is about 1.5 times more often.

**Why.** The router is a simple linear function of the token vector it receives, logits = x·W_g, so tokens with similar vectors get similar logits and the same top expert. The vector that reaches the MoE layer is **not** the raw word embedding: it is the output of the self-attention layer (x + attention(x)), in which every token has just been replaced by a weighted mixture that includes information from the other tokens of the article, mostly from its neighbours. Two neighbouring tokens therefore see largely the same context and receive largely the same mixed-in information, so their vectors move towards each other and are routed alike. In addition, the position embedding added before attention is similar for adjacent positions, and neighbouring words belong to the same phrase, sentence and topic and often have a similar part of speech. The router is therefore not choosing on the identity of the word alone, but on a context-smoothed vector, which creates runs of consecutive tokens on the same expert. This locality has only a small effect on capacity: the diagnostic cell in Task 4 shows that the real routing overflows only about half a point more than independent routing with the same usage.


---
## Submission checklist

- **Runtime → Restart and run all** finishes without errors, with `mini_agnews.tsv` beside the notebook.
- Every task prints the output shown in its *Output your code must print* box.
- Q1–Q5 are answered in the cell above, in your own words, using **your** numbers.
- Submit the `.ipynb` with all outputs saved.

### Common mistakes

- **Adding noise at evaluation time.** The noise is for training only — check `self.training`.
- **Running every expert on every token** and multiplying by zero. It gives the right numbers but it
  is not a sparse layer, and Task 1 asks for sparse dispatch.
- **`gates` rows that do not sum to 1** — usually a softmax over all logits instead of over the
  KeepTopK vector.
- **Forgetting `self.probs` / `self.top_idx` / `self.top1`** in `MoELayer.forward`; Tasks 2–5 read them.
- **Copying the router weights after training** (or without `.clone()`), which makes every router
  look as if it never moved.
- **Computing dropped tokens over the whole test set at once** instead of batch by batch — capacity is
  a per-batch limit.
- **Pairing tokens across two different articles** in Task 5(c).